# Secure Service Account (SSA) Setup + 3LO Token Minting

This notebook covers APS SSA Task 1 and prepares you for ACC provisioning:
1. Get admin 2LO token (`client_credentials`).
2. Create service account (returns generated SSA email).
3. Create service-account key (returns `kid` + private key once).
4. Mint runtime SSA-backed 3LO token (`jwt-bearer`).

Use this once for setup, then store outputs in your secret manager.


## Prerequisites

- APS app type: **Server-to-Server**
- Install dependencies in your environment:

```bash
pip install requests pyjwt[crypto]
```

- Set `CLIENT_ID_SSA` and `CLIENT_SECRET_SSA` from your Server-to-Server app.
- Legacy aliases `APS_SSA_CLIENT_ID` and `APS_SSA_CLIENT_SECRET` also work.


In [1]:
import os
import time
import requests
import jwt

from dotenv import load_dotenv

load_dotenv()

SSA_CLIENT_ID = os.getenv("CLIENT_ID_SSA") or os.getenv("APS_SSA_CLIENT_ID") or os.getenv("APS_CLIENT_ID") or os.getenv("CLIENT_ID")
SSA_CLIENT_SECRET = os.getenv("CLIENT_SECRET_SSA") or os.getenv("APS_SSA_CLIENT_SECRET") or os.getenv("APS_CLIENT_SECRET") or os.getenv("CLIENT_SECRET")

if not SSA_CLIENT_ID or not SSA_CLIENT_SECRET:
    raise RuntimeError("Missing CLIENT_ID_SSA/CLIENT_SECRET_SSA (or APS_SSA_CLIENT_ID/APS_SSA_CLIENT_SECRET fallback).")

BASE_AUTH_URL = "https://developer.api.autodesk.com/authentication/v2"
TOKEN_URL = f"{BASE_AUTH_URL}/token"

ADMIN_SCOPES = [
    "application:service_account:read",
    "application:service_account:write",
    "application:service_account_key:write",
]

RUNTIME_SCOPES = [
    "code:all",
    "data:read",
    "data:write",
    "data:create",
    "data:search",
]

print("Loaded credentials and constants.")


Loaded credentials and constants.


In [2]:
def get_admin_token() -> str:
    resp = requests.post(
        TOKEN_URL,
        data={
            "grant_type": "client_credentials",
            "scope": " ".join(ADMIN_SCOPES),
        },
        auth=(SSA_CLIENT_ID, SSA_CLIENT_SECRET),
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()["access_token"]
admin_token = get_admin_token()
print("Admin token acquired.")


Admin token acquired.


In [ ]:
# Set this to True when you intentionally want to create new SSA resources.
RUN_CREATE = False

SSA_NAME = "service-mycompany-filesync"
FIRST_NAME = "service"
LAST_NAME = "mycompany-filesync"

def create_service_account(token: str) -> dict:
    resp = requests.post(
        f"{BASE_AUTH_URL}/service-accounts",
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/json",
            "Content-Type": "application/json",
        },
        json={
            "name": SSA_NAME,
            "firstName": FIRST_NAME,
            "lastName": LAST_NAME,
        },
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()

if not RUN_CREATE:
    raise RuntimeError("Set RUN_CREATE=True to create a new service account and key.")

service_account = create_service_account(admin_token)
SERVICE_ACCOUNT_ID = service_account["serviceAccountId"]
SSA_EMAIL = service_account["email"]

print("SERVICE_ACCOUNT_ID:", SERVICE_ACCOUNT_ID)
print("SSA_EMAIL:", SSA_EMAIL)


RuntimeError: Set RUN_CREATE=True to create a new service account and key.

In [ ]:
def create_service_account_key(token: str, service_account_id: str) -> dict:
    resp = requests.post(
        f"{BASE_AUTH_URL}/service-accounts/{service_account_id}/keys",
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/json",
        },
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()

key_data = create_service_account_key(admin_token, SERVICE_ACCOUNT_ID)
KEY_ID = key_data["kid"]
PRIVATE_KEY = key_data["privateKey"]

print("KEY_ID:", KEY_ID)
print("PRIVATE_KEY received (store securely).")


In [ ]:
# Save this output in your secrets manager; private key is returned only once.
env_private_key = PRIVATE_KEY.replace("\n", "\\n")
print("\n--- Copy to .env or secret manager ---")
print(f"CLIENT_ID_SSA={SSA_CLIENT_ID}")
print(f"CLIENT_SECRET_SSA={SSA_CLIENT_SECRET}")
print(f"APS_SSA_SERVICE_ACCOUNT_ID={SERVICE_ACCOUNT_ID}")
print(f"APS_SSA_KEY_ID={KEY_ID}")
print(f"APS_SSA_SCOPE={' '.join(RUNTIME_SCOPES)}")
print(f"APS_SSA_EMAIL={SSA_EMAIL}")
print(f"APS_SSA_PRIVATE_KEY=\"{env_private_key}\"")


In [ ]:
def build_ssa_jwt(client_id: str, service_account_id: str, key_id: str, private_key: str, scopes: list[str], exp_seconds: int = 300) -> str:
    if exp_seconds < 1 or exp_seconds > 300:
        raise ValueError("exp_seconds must be between 1 and 300")
    now = int(time.time())
    claims = {
        "iss": client_id,
        "sub": service_account_id,
        "aud": TOKEN_URL,
        "exp": now + exp_seconds,
        "scope": scopes,
    }
    headers = {
        "alg": "RS256",
        "kid": key_id,
    }
    return jwt.encode(claims, private_key, algorithm="RS256", headers=headers)
def mint_runtime_3lo_token(client_id: str, client_secret: str, assertion: str, scopes: list[str]) -> str:
    resp = requests.post(
        TOKEN_URL,
        headers={
            "Accept": "application/json",
            "Content-Type": "application/x-www-form-urlencoded",
        },
        data={
            "grant_type": "urn:ietf:params:oauth:grant-type:jwt-bearer",
            "assertion": assertion,
            "scope": " ".join(scopes),
        },
        auth=(client_id, client_secret),
        timeout=30,
    )
    resp.raise_for_status()
    payload = resp.json()
    if str(payload.get("token_type", "")).lower() != "bearer":
        raise RuntimeError(f"Unexpected token response: {payload}")
    return payload["access_token"]
assertion = build_ssa_jwt(
    client_id=SSA_CLIENT_ID,
    service_account_id=SERVICE_ACCOUNT_ID,
    key_id=KEY_ID,
    private_key=PRIVATE_KEY,
    scopes=RUNTIME_SCOPES,
)
token3lo = mint_runtime_3lo_token(SSA_CLIENT_ID, SSA_CLIENT_SECRET, assertion, RUNTIME_SCOPES)
print("Runtime 3LO token minted. Prefix:", token3lo[:20] + "...")


## ACC Hub provisioning (Task 2)

Use `SSA_EMAIL` from this notebook in ACC:
1. ACC Account Admin -> Custom Integrations: add your Server-to-Server app client id (`CLIENT_ID_SSA`).
2. Invite `SSA_EMAIL` to the target project.
3. Grant project/folder permissions (read/write/version/create).
4. Use the runtime token for ACC + Design Automation flows.
